In [5]:
import sys
from pathlib import Path
from pydantic import BaseModel, Field
from typing import Literal, Optional, List
import json

# Jupyter may start in either the repository root or the notebooks folder.
ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "Scripts" / "trainValSplit.py").is_file()),
    None,
)   
if ROOT is None:
    raise FileNotFoundError("Could not find the project root containing Scripts/trainValSplit.py")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# LOADING MODEL,Processor 

import torch
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
from Scripts import dataUtils as dU
from Scripts import trainValSplit
MODEL_PATH = r"C:\models\HF\Qwen2.5-VL-3B-Instruct"

# Three uncapped manga pages create thousands of visual tokens. Keep each
# page to at most 256 visual tokens (after Qwen's spatial merge).
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)

In [6]:
import trainValSplit
import dataUtils as dU

img_path = r'C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\TaskData\development\images'
json_path = r'C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\TaskData\development\augmented_labels.jsonl'

seqs = dU.json_load_augmented(json_path).keys()
seqs = list(seqs)

X, y = trainValSplit.fit_seq(seqs, 0.05)
train_text = """
Treat the three manga images as one continuous story and process them in the order provided.

Read each manga page in Japanese reading order: panels from right to left and top to bottom, and text within each panel from right to left and top to bottom.

Transcribe every dialogue and narration text exactly as shown. Assign recurring characters consistent labels such as "character1", "character2", and so on across all three images. Use "UNKNOWN" when the speaker cannot be identified and "NARRATION" for narration or captions.

Return only the target JSON array: one inner list per input image, in the same order as the images. Within each inner list, return objects containing only "speaker" and "text" in reading order. If an image has no dialogue or narration, return an empty list for that image.

Output valid JSON only, with no Markdown or explanation.
"""

prompt_builder = dU.MangaPromptBuilder(
    train_text=train_text
)


train_dataset = dU.MangaSequenceDataset(
    label_jsonl_path=json_path,
    image_root=img_path,
    sequence_ids=X
)


train_collator = dU.MangaCollator(
    processor=processor,
    vision_info=process_vision_info,
    prompt_builder=prompt_builder,
    train_mode=True,
    return_prompts=True
)

# inference_collator = dU.MangaCollator(
#     processor=processor,
#     vision_info=process_vision_info,
#     prompt_builder=prompt_builder,
#     train_mode=False,
#     return_metadata=True,
# )



In [4]:
sample1 = train_dataset[0]
sample2 = train_dataset[1]

batch = [sample1, sample2]

inputs,prompts = train_collator.__call__(batch)
print(prompts[0])

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|>
Given these three manga images as one continuous story:

1. Read the pages in order.
2. Transcribe all speaking text.
3. Identify which character speaks each transcript.
4. Use NARRATION for narration.
5. Return only valid JSON.
<|im_end|>
<|im_start|>assistant
{"sequence_id": "seq_71033fec4de494fcPERM0", "pages": [[{"speaker": "character1", "text": "WOAHH!! ...THE SNAKE!!?"}, {"speaker": "character1", "text": "TAKE IT OFF! TAKE IT OFF!!"}, {"speaker": "character1", "text": "THIS IS BAD! IT'S GOING TO BITE ME!"}, {"speaker": "character1", "text": "WAAAAAHHH"}, {"speaker": "character2", "text": "NANKURU-NAISA."}, {"speaker": "character3", "text": "HAHA, YOU SURE SCARE EASY KID~ BUT THAT'S A GOOD THING."}, {"speaker": "character3", "text": "REAL VIPERS ARE ACTUALLY VERY DANGEROUS."}, {"